## PropertyLens RAG Notebook (Pinecone hybrid + Ollama Gemma 3)

This notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline for PropertyLens:

**Data → chunking → Pinecone hybrid index → hybrid retrieval (dense+sparse) → RRF fusion → cross-encoder rerank → MMR diversity → lost-in-the-middle reorder → multi-query retrieval → answer generation with Ollama + Gemma 3**.

### Architecture (conceptual)
- **Ingest**: transaction + trend “documents” become chunks.
- **Index**: Pinecone hybrid vectors store both:
  - dense embeddings (semantic)
  - sparse BM25 weights (keyword)
- **Retrieve**:
  - dense-only list (alpha=1)
  - sparse-only list (alpha=0)
  - fuse with **Reciprocal Rank Fusion (RRF)**
- **Rerank**: cross-encoder (`BAAI/bge-reranker-v2-m3`)
- **Diversify**: MMR
- **Order**: lost-in-the-middle mitigation (best chunk first + second-best last)
- **Extra recall**: multi-query retrieval (Gemma 3 generates reformulations)
- **Generate**: Gemma 3 answers strictly from retrieved context

### Prerequisites
- A repo-root `.env` with `PINECONE_API_KEY=...`
- Optional: `GEMINI_API_KEY=...` for the **Google Gemini** answer cell (cloud LLM)
- Pinecone account access
- Ollama running locally, with Gemma 3 pulled:
  - `ollama pull gemma3`
  - Ollama server listening on `http://localhost:11434`

Notes:
- Full ingest over all CSVs can be slow/large. Start with a small sample first.


In [1]:
# This is the ONLY cell that uses !pip install
# It installs every dependency used in the entire notebook.
#
# pinecone + pinecone-text       -> Pinecone client + BM25 sparse encoder
# sentence-transformers          -> BGE-M3 dense embeddings
# transformers + torch           -> cross-encoder reranker (BAAI/bge-reranker-v2-m3)
# langchain + langchain-community-> multi-query helper utilities (optional)
# ollama                         -> local Gemma 3 LLM client
# google-generativeai            -> Google Gemini API (optional LLM for answers)
# pandas + numpy + tqdm          -> data processing

!pip install pinecone pinecone-text sentence-transformers transformers torch \
             langchain langchain-community ollama google-generativeai \
             pandas numpy tqdm python-dotenv


### Configuration and environment setup

All secrets and tunable parameters are centralized here.

- Create a repo-root `.env` with `PINECONE_API_KEY=...` (and optionally `GEMINI_API_KEY=...` for the Gemini answer cell)
- You can tune retrieval sizes (`TOP_K_*`), RRF constant (`RRF_K`), and MMR diversity (`MMR_LAMBDA`).
- The notebook expects data under `../data` relative to `notebooks/`.

**Inputs**: `.env`

**Outputs**: in-memory config constants used by later cells.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --- Pinecone ---
PINECONE_API_KEY   = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX     = "propertylens-rag"
PINECONE_NAMESPACE = "hdb-v1"

# --- Embedding model (dense) ---
DENSE_MODEL_NAME   = "BAAI/bge-m3"
DENSE_DIMENSION    = 1024

# --- Reranker ---
RERANKER_MODEL     = "BAAI/bge-reranker-v2-m3"

# --- Retrieval knobs ---
TOP_K_RETRIEVAL    = 50   # candidates from hybrid search
TOP_K_RERANK       = 10   # after cross-encoder
TOP_K_MMR          = 5    # after MMR diversity filter
TOP_K_FINAL        = 5    # sent to LLM context window
MMR_LAMBDA         = 0.7  # 1.0 = pure relevance, 0.0 = pure diversity
RRF_K              = 60   # standard RRF constant

# --- Multi-query ---
N_SUBQUERIES       = 3    # number of query reformulations

# --- LLM ---
OLLAMA_MODEL       = "gemma3"
OLLAMA_BASE_URL    = "http://localhost:11434"

# --- Gemini (optional cloud LLM; set GEMINI_API_KEY in repo-root .env) ---
GEMINI_API_KEY     = os.getenv("GEMINI_API_KEY", "").strip()
GEMINI_MODEL       = os.getenv("GEMINI_MODEL", "gemini-2.5-flash").strip()


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repo root by searching upward for a `data/` directory.

    Args:
        start: starting directory (defaults to current working directory).

    Returns:
        Path to repo root.

    Raises:
        FileNotFoundError: if no repo root could be inferred.
    """
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Could not locate repo root. Expected to find `data/` and `notebooks/` directories above current working directory."
    )


REPO_ROOT = find_repo_root()

# --- Data paths (absolute -> robust regardless of notebook launch dir) ---
DATA_ROOT          = str(REPO_ROOT / "data")
TRANSACTIONS_GLOB  = str(Path(DATA_ROOT) / "feature_data" / "**" / "outputs" / "*.csv")
AMENITIES_GLOB     = str(Path(DATA_ROOT) / "amenities" / "*.csv")
ARTIFACTS_DIR      = str(Path(DATA_ROOT) / "artifacts")

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY. Add it to repo-root .env"
print("Loaded config OK")
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)


Loaded config OK
REPO_ROOT: /Users/bhuvesh/Documents/PropertyLens
DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/data


### Load and preview raw data

We load:
- **Transactions**: all CSVs matching `TRANSACTIONS_GLOB` (feature-layer outputs). Some files may be EDA outputs; we filter to the likely HDB feature tables.
- **Amenities**: `data/amenities/*.csv`.
- **Trends**: derived from transactions as median resale price per `town` per `year`.

Expected transaction fields vary by file. This notebook tries to normalize:
- `resale_price`
- `transaction_year` (or `year`)
- `address_key` (optional)
- `town` (either direct `town` column, or inferred from `town_*` one-hots)
- `flat_type` (either direct, or inferred from `flat_type_*` one-hots)

**Outputs**: `transactions_df`, `amenities_df`, `trends_df`


In [3]:
from __future__ import annotations

import glob
from typing import Optional

import pandas as pd


def _infer_from_onehots(df: pd.DataFrame, prefix: str) -> Optional[pd.Series]:
    """Infer a categorical value from one-hot columns like town_* or flat_type_*.

    Args:
        df: Source DataFrame.
        prefix: Column prefix, e.g. "town_".

    Returns:
        A pandas Series of inferred category values, or None if no matching columns.
    """
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return None
    # idxmax gives the column name; strip prefix
    return df[cols].idxmax(axis=1).str.replace(prefix, "", regex=False)


def load_transactions(glob_pattern: str) -> pd.DataFrame:
    """Load all HDB transaction CSVs matching glob_pattern into one DataFrame.

    This function is defensive: it filters out non-transaction CSVs by checking for
    key columns (e.g., resale_price + transaction_year/year).

    Args:
        glob_pattern: Glob for transaction CSVs.

    Returns:
        Normalized DataFrame with at least: town, flat_type, transaction_year, resale_price.
    """
    paths = sorted(glob.glob(glob_pattern, recursive=True))
    if not paths:
        raise FileNotFoundError(f"No CSVs found for {glob_pattern}")

    frames: list[pd.DataFrame] = []
    for p in paths:
        # Light filter: skip EDA profiles/ablation summaries
        name = p.lower()
        if "eda_profiles" in name or "ablation" in name or "column_profile" in name:
            continue

        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        if "resale_price" not in df.columns:
            continue
        year_col = "transaction_year" if "transaction_year" in df.columns else ("year" if "year" in df.columns else None)
        if year_col is None:
            continue

        if "town" not in df.columns:
            town = _infer_from_onehots(df, "town_")
            if town is not None:
                df["town"] = town

        if "flat_type" not in df.columns:
            ft = _infer_from_onehots(df, "flat_type_")
            if ft is not None:
                df["flat_type"] = ft

        df["transaction_year"] = df[year_col].astype("int", errors="ignore")
        df["town"] = df.get("town", "").astype(str).str.upper()
        df["flat_type"] = df.get("flat_type", "").astype(str).str.upper()

        keep = [c for c in ["address_key", "town", "flat_type", "transaction_year", "resale_price", "floor_area_sqm", "storey_range", "level_mid"] if c in df.columns]
        frames.append(df[keep].copy())

    if not frames:
        raise ValueError("No usable transaction tables found (missing required columns)")

    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=["resale_price", "transaction_year"]).copy()
    out["resale_price"] = pd.to_numeric(out["resale_price"], errors="coerce")
    out = out.dropna(subset=["resale_price"]).copy()
    return out


def load_amenities(glob_pattern: str) -> pd.DataFrame:
    """Load all amenity CSVs into one DataFrame.

    Args:
        glob_pattern: Glob for amenity CSVs.

    Returns:
        DataFrame with at least: name, lat, lng, type/category if available.
    """
    paths = sorted(glob.glob(glob_pattern, recursive=True))
    if not paths:
        raise FileNotFoundError(f"No amenity CSVs found for {glob_pattern}")

    frames: list[pd.DataFrame] = []
    for p in paths:
        df = pd.read_csv(p)
        if "name" not in df.columns:
            continue
        df["source_file"] = os.path.basename(p)
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    for c in ("lat", "lng"):
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def derive_trends(transactions: pd.DataFrame) -> pd.DataFrame:
    """Compute median resale_price per town per year from transaction data.

    Args:
        transactions: Transaction DataFrame.

    Returns:
        Trends DataFrame with columns: town, year, median_resale_price, transaction_count.
    """
    df = transactions.copy()
    df = df[df["town"].astype(str).str.len() > 0]
    agg = (
        df.groupby(["town", "transaction_year"], dropna=False)
        .agg(
            median_resale_price=("resale_price", "median"),
            transaction_count=("resale_price", "count"),
        )
        .reset_index()
        .rename(columns={"transaction_year": "year"})
        .sort_values(["town", "year"])
    )
    agg["median_resale_price"] = agg["median_resale_price"].round(0)
    return agg


# --- Run ---
transactions_df = load_transactions(TRANSACTIONS_GLOB)
amenities_df    = load_amenities(AMENITIES_GLOB)
trends_df       = derive_trends(transactions_df)

print(f"Transactions: {len(transactions_df):,} rows")
print(f"Amenities:    {len(amenities_df):,} rows")
print(f"Trends:       {len(trends_df):,} rows")
transactions_df.head(3)


Transactions: 1,568,804 rows
Amenities:    615 rows
Trends:       312 rows


,address_key,town,flat_type,transaction_year,resale_price,floor_area_sqm,level_mid
0,174 ANG MO KIO AVE 4,ANG MO KIO,3 ROOM,2015,255000.0,60.0,8.0
1,541 ANG MO KIO AVE 10,ANG MO KIO,3 ROOM,2015,275000.0,68.0,2.0
2,163 ANG MO KIO AVE 4,ANG MO KIO,3 ROOM,2015,285000.0,69.0,2.0


### Build town-level amenity summary

Because transaction records do not carry GPS coordinates, we join amenities at the
**town level** rather than by proximity radius.

Steps:
1. Normalise town names in both DataFrames to uppercase stripped strings.
2. Group the amenities DataFrame by `town` and `source_file` (amenity type).
3. For each town, build a compact summary string listing counts per amenity type
   and up to 3 example names per type.

**Output**: `town_amenity_summary` — a `dict[str, str]` mapping UPPERCASE town name
to a multi-line amenity summary string ready to be appended to any chunk in that town.


In [4]:
from __future__ import annotations

import pandas as pd


def _normalise_town(name: str) -> str:
    """Uppercase and strip a town name for consistent matching."""
    return str(name).upper().strip()


def _summarise_amenity_type(group: pd.DataFrame, max_examples: int = 3) -> str:
    """
    Return a one-line summary for one amenity type group.

    Args:
        group: rows from the amenities DataFrame for one (town, source_file) pair.
        max_examples: max number of example names to include.

    Returns:
        Summary string, e.g. "MRT stations (4): Tampines, Simei, Pasir Ris, ..."
    """
    amenity_type = str(group["source_file"].iloc[0]).replace(".csv", "").replace("_", " ").title()
    count = len(group)
    names = group["name"].dropna().astype(str).tolist()
    examples = ", ".join(names[:max_examples])
    suffix = ", ..." if count > max_examples else ""
    return f"{amenity_type} ({count}): {examples}{suffix}"


def build_town_amenity_summary(amenities: pd.DataFrame) -> dict[str, str]:
    """
    Build a town → amenity summary string mapping.

    Groups amenities by town and amenity type, then formats a compact
    multi-line summary for each town.

    Args:
        amenities: amenities DataFrame with at least 'town', 'name', 'source_file' columns.

    Returns:
        dict mapping uppercase town name to a formatted amenity summary string.
        Towns with no amenities are not included.
    """
    if "town" not in amenities.columns:
        print("WARNING: amenities DataFrame has no 'town' column — summary will be empty.")
        return {}

    amenities = amenities.copy()
    amenities["town"] = amenities["town"].apply(_normalise_town)

    summary: dict[str, str] = {}

    for town, town_group in amenities.groupby("town"):
        lines: list[str] = []

        if "source_file" in town_group.columns:
            for src, src_group in town_group.groupby("source_file"):
                lines.append(_summarise_amenity_type(src_group))
        else:
            # Fallback: no source_file column, just count total
            names = town_group["name"].dropna().astype(str).tolist()
            examples = ", ".join(names[:5])
            suffix = ", ..." if len(names) > 5 else ""
            lines.append(f"Amenities ({len(names)}): {examples}{suffix}")

        summary[str(town)] = "\n".join(lines)

    return summary


def _infer_town_from_address(addr: str, town_names: tuple[str, ...]) -> str:
    """Best-effort town label from a free-text address (amenity CSVs often lack `town`)."""
    u = str(addr).upper()
    for t in sorted(town_names, key=len, reverse=True):
        if t in u:
            return t
    return ""


def _ensure_amenity_town_column(amenities: pd.DataFrame) -> pd.DataFrame:
    """Add `town` when missing, by matching HDB town names inside `address`."""
    out = amenities.copy()
    if "town" in out.columns and out["town"].notna().any():
        return out
    towns = (
        "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT BATOK", "BUKIT MERAH",
        "BUKIT PANJANG", "BUKIT TIMAH", "CENTRAL AREA", "CHOA CHU KANG",
        "CLEMENTI", "GEYLANG", "HOUGANG", "JURONG EAST", "JURONG WEST",
        "KALLANG/WHAMPOA", "MARINE PARADE", "PASIR RIS", "PUNGGOL",
        "QUEENSTOWN", "SEMBAWANG", "SENGKANG", "SERANGOON", "TAMPINES",
        "TOA PAYOH", "WOODLANDS", "YISHUN",
    )
    if "address" not in out.columns:
        print("WARNING: no `town` or `address` on amenities — cannot infer town.")
        out["town"] = ""
        return out
    out["town"] = out["address"].map(lambda a: _infer_town_from_address(a, towns))
    return out


# --- Run ---
amenities_with_town = _ensure_amenity_town_column(amenities_df)
town_amenity_summary = build_town_amenity_summary(amenities_with_town)

print(f"Towns with amenity summaries: {len(town_amenity_summary)}")
if town_amenity_summary:
    sample_town = next(iter(town_amenity_summary))
    print(f"\nExample — {sample_town}:\n{town_amenity_summary[sample_town]}")
else:
    print("\n(No summaries — add `town` to amenities or ensure `address` contains a known town name.)")


Towns with amenity summaries: 25

Example — :
Hawker Centres (69): HAWKER CENTRE (BLK 1 JALAN KUKOH), HAWKER CENTRE (BLK 208B NEW UPPER CHANGI ROAD), JALAN BATU HAWKER CENTRE, ...
Malls (94): ION ORCHARD, NGEE ANN CITY, PARAGON SINGAPORE, ...
Mrt Stations (92): CANBERRA MRT STATION (NS12), NOVENA MRT STATION (NS20), NEWTON MRT STATION, ...
Schools (68): FAIRFIELD METHODIST SCHOOL (PRIMARY), HENRY PARK PRIMARY SCHOOL, HOLY INNOCENTS' PRIMARY SCHOOL, ...


### Build text chunks from transactions (parent–child)

We use a parent–child chunking strategy:
- **Child chunk (~256 tokens)**: core facts only (town, flat type, storey, floor area, resale price, derived psf, year).
- **Parent chunk (~1024 tokens)**: child facts + contextual enrichment.

Parent chunks append a **town-level amenity summary** (from `town_amenity_summary`) so the LLM sees malls, MRT, schools, etc. for that town without GPS on transactions.

Each chunk includes 1–2 sentence **situating context** up front (helps BM25).

**Outputs**: list of chunk dicts (id, text, parent_text, metadata) suitable for Pinecone upsert.


In [5]:
from __future__ import annotations

import math
import re
from typing import Any

import numpy as np
import pandas as pd


def _bucket_storey(storey_range: str | float | int | None) -> str:
    """Bucket a storey_range string into a coarse band."""
    if storey_range is None or (isinstance(storey_range, float) and math.isnan(storey_range)):
        return "unknown"
    s = str(storey_range).upper().strip()
    # Attempt to parse "07 TO 09" style
    m = re.search(r"(\d{1,2})\s*TO\s*(\d{1,2})", s)
    if m:
        lo, hi = int(m.group(1)), int(m.group(2))
        mid = (lo + hi) / 2
    else:
        # Fallback: parse first number
        m2 = re.search(r"(\d{1,2})", s)
        mid = float(m2.group(1)) if m2 else None

    if mid is None:
        return "unknown"
    if mid <= 5:
        return "01-05"
    if mid <= 12:
        return "06-12"
    if mid <= 20:
        return "13-20"
    return "21+"


def _bucket_price(price: float) -> str:
    """Bucket prices into 100k bands for metadata filtering."""
    if not np.isfinite(price) or price <= 0:
        return "unknown"
    lo = int(price // 100000) * 100
    hi = lo + 100
    return f"{lo}k-{hi}k"


def _safe_float(x: Any) -> float | None:
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def format_child_chunk(row: pd.Series) -> str:
    """Format a transaction row into a short child chunk string (~256 tokens)."""
    town = str(row.get("town", "")).upper()
    flat_type = str(row.get("flat_type", "")).upper()
    year = int(row.get("transaction_year", 0) or 0)
    price = _safe_float(row.get("resale_price")) or 0.0
    area = _safe_float(row.get("floor_area_sqm"))
    psf = (price / area / 10.7639) if area and area > 0 else None
    storey_band = _bucket_storey(row.get("storey_range"))

    ctx = f"This is an HDB resale transaction in {town}, year {year}."
    facts = [
        f"Town: {town}",
        f"Flat type: {flat_type}",
        f"Storey band: {storey_band}",
        f"Floor area (sqm): {area:.1f}" if area else "Floor area (sqm): unknown",
        f"Resale price (SGD): {int(round(price))}",
        f"Approx PSF (SGD): {psf:.1f}" if psf else "Approx PSF (SGD): unknown",
    ]
    return ctx + "\n" + "\n".join(facts)


def format_parent_chunk(row: pd.Series, amenity_summary: str) -> str:
    """
    Format a transaction row + town-level amenity summary into a rich parent chunk.

    Args:
        row: one transaction row.
        amenity_summary: pre-built amenity summary string for this town
                         (from town_amenity_summary dict). Empty string if unavailable.

    Returns:
        Rich parent chunk string (~1024 tokens) for LLM context.
    """
    base = format_child_chunk(row)
    lines = [base, "", "Nearby amenities (town-level):"]

    if amenity_summary:
        lines.append(amenity_summary)
    else:
        lines.append("- No amenity data available for this town.")

    return "\n".join(lines)


def build_chunks(
    transactions: pd.DataFrame,
    town_amenity_summary: dict[str, str],
) -> list[dict]:
    """
    Build a list of chunk dicts ready for Pinecone upsert.

    Uses town-level amenity join (no GPS coordinates required).
    Each chunk has:
    - id: stable unique id
    - text: child chunk (embedded + BM25 fitted on this)
    - parent_text: child + town amenity summary (sent to LLM)
    - metadata: filterable fields

    Args:
        transactions: HDB transactions DataFrame.
        town_amenity_summary: dict mapping uppercase town name to amenity summary string.

    Returns:
        List of chunk dicts.
    """
    out: list[dict] = []

    for i, row in transactions.reset_index(drop=True).iterrows():
        town       = str(row.get("town", "")).upper().strip()
        flat_type  = str(row.get("flat_type", "")).upper()
        year       = int(row.get("transaction_year", 0) or 0)
        price      = _safe_float(row.get("resale_price")) or 0.0
        area       = _safe_float(row.get("floor_area_sqm"))
        psf        = (price / area / 10.7639) if area and area > 0 else None

        address_key = str(row.get("address_key", "")) if "address_key" in row else ""
        base_id     = address_key.strip().upper().replace(" ", "_") if address_key else f"row_{i}"
        chunk_id    = f"txn_{base_id}_{year}"

        # Town-level amenity join — no coordinates needed
        amenity_summary = town_amenity_summary.get(town, "")

        child  = format_child_chunk(row)
        parent = format_parent_chunk(row, amenity_summary)

        out.append(
            {
                "id": chunk_id,
                "text": child,
                "parent_text": parent,
                "metadata": {
                    "town":        town,
                    "flat_type":   flat_type,
                    "storey_band": _bucket_storey(row.get("storey_range")),
                    "sale_year":   year,
                    "price_band":  _bucket_price(price),
                    "resale_price": int(round(price)),
                    "psf":         float(round(psf, 1)) if psf is not None else None,
                    "source":      "transaction",
                },
            }
        )

    return out


chunks = build_chunks(transactions_df, town_amenity_summary)
print(f"Built {len(chunks):,} chunks")
print("\nExample child chunk:\n", chunks[0]["text"])
print("\nExample parent chunk:\n", chunks[0]["parent_text"][:800], "...")


Built 1,568,804 chunks

Example child chunk:
 This is an HDB resale transaction in ANG MO KIO, year 2015.
Town: ANG MO KIO
Flat type: 3 ROOM
Storey band: unknown
Floor area (sqm): 60.0
Resale price (SGD): 255000
Approx PSF (SGD): 394.8

Example parent chunk:
 This is an HDB resale transaction in ANG MO KIO, year 2015.
Town: ANG MO KIO
Flat type: 3 ROOM
Storey band: unknown
Floor area (sqm): 60.0
Resale price (SGD): 255000
Approx PSF (SGD): 394.8

Nearby amenities (town-level):
Hawker Centres (7): MARKET & HAWKER CENTRE (BLK 409 ANG MO KIO AVE 10), MARKET & HAWKER CENTRE (BLK 724 ANG MO KIO AVE 6), CHENG SAN MARKET AND COOKED FOOD CENTRE, ...
Malls (2): AMK HUB, BROADWAY PLAZA ANG MO KIO
Mrt Stations (2): ANG MO KIO MRT STATION, MAYFLOWER MRT STATION (TE6)
Schools (7): ANDERSON PRIMARY SCHOOL, MAYFLOWER PRIMARY SCHOOL, JING SHAN PRIMARY SCHOOL, ... ...


### Build trend chunks

We store trends as separate chunks so the LLM can answer questions like:
- “Are prices rising in Tampines?”

Each row in the trends table becomes one chunk with a human-readable summary.

**Outputs**: `trend_chunks` (same chunk dict shape as transactions).


In [6]:
from __future__ import annotations

from typing import Any

import pandas as pd


def build_trend_chunks(trends: pd.DataFrame) -> list[dict[str, Any]]:
    """Convert the trends DataFrame into one chunk per town-year combination.

    Args:
        trends: Trends DataFrame with town, year, median_resale_price, transaction_count.

    Returns:
        List of chunk dicts.
    """
    out: list[dict[str, Any]] = []
    for _, row in trends.iterrows():
        town = str(row.get("town", "")).upper()
        year = int(row.get("year", 0) or 0)
        med = float(row.get("median_resale_price", 0) or 0)
        n = int(row.get("transaction_count", 0) or 0)

        text = (
            f"HDB resale trend summary for {town}, year {year}: "
            f"median resale price is SGD {int(round(med)):,} across {n:,} transactions."
        )

        out.append(
            {
                "id": f"trend_{town}_{year}",
                "text": text,
                "parent_text": text,
                "metadata": {
                    "town": town,
                    "flat_type": "",
                    "storey_band": "",
                    "sale_year": year,
                    "price_band": "",
                    "resale_price": int(round(med)),
                    "psf": None,
                    "source": "trend",
                },
            }
        )
    return out


trend_chunks = build_trend_chunks(trends_df)
print(f"Built {len(trend_chunks):,} trend chunks")
print("\nExample:\n", trend_chunks[0]["text"])


Built 312 trend chunks

Example:
 HDB resale trend summary for ANG MO KIO, year 2015: median resale price is SGD 355,000 across 5,390 transactions.


### Initialise embedding models

We use two encoders:
1) **Dense** bi-encoder: `BAAI/bge-m3` (1024-dim) for semantic similarity.
2) **Sparse** BM25 encoder: `BM25Encoder` from `pinecone-text` for keyword matching.

Important:
- Both encoders must be applied to the **same text** field (`chunk['text']`) at ingest and query time.
- BM25 must be **fitted** on the corpus before use. We persist it to disk to avoid refitting.

**Outputs**: `dense_encoder`, `bm25_encoder`


In [7]:
from __future__ import annotations

import pickle
from pathlib import Path

from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def load_dense_encoder(model_name: str) -> SentenceTransformer:
    """Load BGE-M3 dense encoder."""
    return SentenceTransformer(model_name)


def fit_bm25_encoder(corpus_texts: list[str], cache_path: str = "bm25_encoder.pkl") -> BM25Encoder:
    """Fit a BM25Encoder on the full corpus and cache it.

    Args:
        corpus_texts: list of documents.
        cache_path: local cache file.

    Returns:
        Fitted BM25Encoder.
    """
    p = Path(cache_path)
    if p.exists():
        with p.open("rb") as f:
            return pickle.load(f)

    enc = BM25Encoder()
    enc.fit(corpus_texts)

    with p.open("wb") as f:
        pickle.dump(enc, f)

    return enc


dense_encoder = load_dense_encoder(DENSE_MODEL_NAME)
corpus_texts  = [c["text"] for c in (chunks + trend_chunks)]
bm25_encoder  = fit_bm25_encoder(corpus_texts)

print("Dense encoder loaded; BM25 fitted.")


/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39174.76it/s]


Dense encoder loaded; BM25 fitted.


### Initialise Pinecone and create index

We use a Pinecone **hybrid** index. Each record stores:
- `values`: dense vector (float list)
- `sparse_values`: BM25 sparse vector (`{'indices': [...], 'values': [...]}`)
- `metadata`: filterable fields and the `parent_text` (the richer text we return to the LLM)

Important:
- The hybrid index must use **`dotproduct`** metric.
- We store metadata fields like `town`, `flat_type`, `sale_year`, and `source` for optional pre-filtering.

**Outputs**: `pc`, `index`


In [8]:
from __future__ import annotations

from pinecone import Pinecone, ServerlessSpec


def init_pinecone(api_key: str) -> Pinecone:
    """Initialise and return a Pinecone client."""
    return Pinecone(api_key=api_key)


def get_or_create_index(pc: Pinecone, index_name: str, dimension: int):
    """Create a Pinecone serverless index if it does not already exist.

    Uses dotproduct metric (required for hybrid search).

    Args:
        pc: Pinecone client.
        index_name: index name.
        dimension: dense vector dimension.

    Returns:
        Pinecone Index object.
    """
    existing = {i["name"] for i in pc.list_indexes()}
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    return pc.Index(index_name)


pc = init_pinecone(PINECONE_API_KEY)
index = get_or_create_index(pc, PINECONE_INDEX, DENSE_DIMENSION)
print(index.describe_index_stats())


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:39:47 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '71',
                                    'x-pinecone-request-latency-ms': '68',
                                    'x-pinecone-response-duration-ms': '73'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 5638},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 251}},
 'storageFullness': 0.0,
 'total_vector_count': 6286,
 'vector_type': 'dense'}


### Encode and upsert chunks

Upsert process:
1. Encode each chunk’s `text` with both encoders
2. Build Pinecone record: `{id, values, sparse_values, metadata}`
3. Upsert in batches (default 100)
4. Store `parent_text` inside metadata so it can be returned at query time

Warning:
- This can be slow/expensive on a full dataset.
- Start with a sample (e.g. first 1000 chunks) and verify retrieval quality.

**Outputs**: upserted vectors in Pinecone.


In [9]:
from __future__ import annotations

from typing import Any

import numpy as np
from tqdm import tqdm


def encode_chunk(
    chunk: dict[str, Any],
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
) -> dict[str, Any]:
    """Encode one chunk into a Pinecone-ready record dict.

    Args:
        chunk: chunk dict with id/text/parent_text/metadata.
        dense_encoder: sentence-transformers model.
        bm25_encoder: fitted BM25 encoder.

    Returns:
        Pinecone record dict.
    """
    text = chunk["text"]
    dense = dense_encoder.encode(text, normalize_embeddings=True)
    dense = dense.astype(np.float32).tolist()

    sparse = bm25_encoder.encode_documents([text])[0]

    md = dict(chunk.get("metadata") or {})
    md["parent_text"] = chunk.get("parent_text", "")

    return {
        "id": chunk["id"],
        "values": dense,
        "sparse_values": sparse,
        "metadata": md,
    }


def upsert_in_batches(
    index,
    records: list[dict[str, Any]],
    namespace: str,
    batch_size: int = 100,
) -> None:
    """Upsert records to Pinecone in batches with a progress bar."""
    for i in tqdm(range(0, len(records), batch_size)):
        batch = records[i : i + batch_size]
        index.upsert(vectors=batch, namespace=namespace)


# --- Upsert (sample by default) ---
all_chunks  = chunks + trend_chunks
sample_n = min(1000, len(all_chunks))
all_chunks_sample = all_chunks[:sample_n]

all_records = [encode_chunk(c, dense_encoder, bm25_encoder) for c in tqdm(all_chunks_sample)]
upsert_in_batches(index, all_records, namespace=PINECONE_NAMESPACE)
print("Upsert complete (sample).")
print(index.describe_index_stats())


100%|██████████| 10/10 [00:11<00:00,  1.14s/it]


Upsert complete (sample).
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '309',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:40:22 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '39',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'hdb-v1': {'vector_count': 1000},
                'transactions': {'vector_count': 5638},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 251}},
 'storageFulln

### Hybrid retrieval (BM25 + cosine)

Pinecone hybrid search blends dense + sparse vectors.

Here we intentionally run retrieval **twice** to get two independent ranked lists:
- **Dense-only** (`alpha=1.0`): semantic similarity
- **Sparse-only** (`alpha=0.0`): BM25 keyword matching

We then fuse lists with RRF.

Notes:
- `alpha` is implemented by scaling: `dense * alpha` and `sparse * (1-alpha)`.
- You can pass Pinecone metadata filters (e.g., `{"town": "TAMPINES"}`) to scope retrieval.


In [10]:
from __future__ import annotations

from typing import Any, Optional


def encode_query_dense(query: str, encoder: SentenceTransformer) -> list[float]:
    """Encode a query string into a dense vector."""
    v = encoder.encode(query, normalize_embeddings=True)
    return v.astype(np.float32).tolist()


def encode_query_sparse(query: str, encoder: BM25Encoder) -> dict[str, Any]:
    """Encode a query string into a sparse BM25 vector."""
    return encoder.encode_queries([query])[0]


def _scale_sparse(sparse: dict[str, Any], scale: float) -> dict[str, Any]:
    """Scale sparse BM25 weights by `scale`."""
    if not sparse or scale == 1.0:
        return sparse
    return {
        "indices": sparse.get("indices", []),
        "values": [float(v) * scale for v in sparse.get("values", [])],
    }


def retrieve_hybrid(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    top_k: int,
    namespace: str,
    alpha: float,
    metadata_filter: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    """Run hybrid retrieval with an explicit alpha.

    Args:
        index: Pinecone Index.
        query: query string.
        dense_encoder: dense encoder.
        bm25_encoder: fitted BM25 encoder.
        top_k: number of matches.
        namespace: Pinecone namespace.
        alpha: 1.0 dense-only, 0.0 sparse-only.
        metadata_filter: optional Pinecone metadata filter.

    Returns:
        List of match dicts.
    """
    dense = encode_query_dense(query, dense_encoder)
    dense = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()

    sparse = encode_query_sparse(query, bm25_encoder)
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))

    res = index.query(
        vector=dense,
        sparse_vector=sparse,
        top_k=int(top_k),
        namespace=namespace,
        include_metadata=True,
        filter=metadata_filter,
    )

    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", None)
    if matches is None:
        return []

    # Normalize to dicts
    out = []
    for m in matches:
        if isinstance(m, dict):
            out.append(m)
        else:
            out.append({
                "id": getattr(m, "id", None),
                "score": getattr(m, "score", None),
                "metadata": getattr(m, "metadata", None),
            })
    return out


def retrieve_dense(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    """Run dense (alpha=1.0) retrieval."""
    return retrieve_hybrid(index, query, dense_encoder, bm25_encoder, top_k, namespace, alpha=1.0, metadata_filter=metadata_filter)


def retrieve_sparse(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    """Run sparse (alpha=0.0) retrieval."""
    return retrieve_hybrid(index, query, dense_encoder, bm25_encoder, top_k, namespace, alpha=0.0, metadata_filter=metadata_filter)


### Reciprocal Rank Fusion (RRF)

RRF merges multiple ranked lists into a single ranking:

\[
score(d) = \sum_i \frac{1}{k + rank_i(d)}
\]

- `rank_i(d)` is 1-based rank position in list *i*.
- `k=60` is a common default to avoid dominance by a single list.

Documents appearing in both dense and sparse lists get higher fused scores.

**Output**: a single fused candidate list with `rrf_score`.


In [11]:
from __future__ import annotations

from typing import Any


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
) -> list[dict[str, Any]]:
    """Merge multiple ranked result lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: list of lists sorted by relevance descending. Each result must have 'id'.
        k: RRF constant.

    Returns:
        Deduplicated results sorted by RRF score desc, with 'rrf_score' added.
    """
    scores: dict[str, float] = {}
    best_match: dict[str, dict[str, Any]] = {}

    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid = str(r.get("id"))
            if not rid:
                continue
            scores[rid] = scores.get(rid, 0.0) + 1.0 / (float(k) + float(rank))
            # keep any representative match payload
            if rid not in best_match:
                best_match[rid] = r

    fused = []
    for rid, sc in scores.items():
        item = dict(best_match[rid])
        item["rrf_score"] = float(sc)
        fused.append(item)

    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


# --- Example usage ---
q0 = "4-room Tampines resale 2024"
dense_results  = retrieve_dense(index, q0, dense_encoder, bm25_encoder, TOP_K_RETRIEVAL, PINECONE_NAMESPACE)
sparse_results = retrieve_sparse(index, q0, dense_encoder, bm25_encoder, TOP_K_RETRIEVAL, PINECONE_NAMESPACE)
fused_results  = reciprocal_rank_fusion([dense_results, sparse_results], k=RRF_K)
print(f"Dense: {len(dense_results)}, Sparse: {len(sparse_results)}, Fused: {len(fused_results)}")
print("Top fused ids:", [r["id"] for r in fused_results[:5]])


Dense: 50, Sparse: 50, Fused: 73
Top fused ids: ['txn_469_TAMPINES_ST_44_2015', 'txn_282_TAMPINES_ST_22_2015', 'txn_241_TAMPINES_ST_21_2015', 'txn_164_TAMPINES_ST_12_2015', 'txn_261_TAMPINES_ST_21_2015']


### Cross-encoder reranker

Retrieval uses a **bi-encoder** (fast, approximate). Reranking uses a **cross-encoder** (slower, more accurate) that scores *(query, passage)* pairs.

We load `BAAI/bge-reranker-v2-m3` using HuggingFace Transformers and rerank the RRF top-50 down to top-10.

**Outputs**: `ce_tokenizer`, `ce_model`, and `reranked` results with a `ce_score`.


In [12]:
from __future__ import annotations

from typing import Any, Tuple

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_cross_encoder(model_name: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model.

    Returns:
        (tokenizer, model)
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    return tok, model


def _candidate_text(c: dict[str, Any], text_field: str) -> str:
    md = c.get("metadata") or {}
    return str(md.get(text_field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
    text_field: str = "parent_text",
) -> list[dict[str, Any]]:
    """Rerank candidates using a cross-encoder.

    Args:
        query: user query.
        candidates: list of candidate dicts.
        tokenizer: HF tokenizer.
        model: HF model.
        top_k: number of outputs.
        text_field: which field in metadata to score.

    Returns:
        Top-k candidates with 'ce_score'.
    """
    if not candidates:
        return []

    pairs = [(query, _candidate_text(c, text_field)) for c in candidates]

    # Batch scoring (small batches to avoid OOM)
    scores: list[float] = []
    bs = 16
    with torch.no_grad():
        for i in range(0, len(pairs), bs):
            batch = pairs[i : i + bs]
            inputs = tokenizer(
                [p[0] for p in batch],
                [p[1] for p in batch],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            out = model(**inputs)
            logits = out.logits.squeeze(-1).detach().cpu().numpy().tolist()
            if isinstance(logits, float):
                logits = [logits]
            scores.extend([float(x) for x in logits])

    scored = []
    for c, s in zip(candidates, scores):
        d = dict(c)
        d["ce_score"] = float(s)
        scored.append(d)

    scored.sort(key=lambda x: x.get("ce_score", -1e9), reverse=True)
    return scored[: int(top_k)]


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL)
reranked = rerank_cross_encoder(
    query="Is $580k fair for a 4-room in Tampines?",
    candidates=fused_results[:TOP_K_RETRIEVAL],
    tokenizer=ce_tokenizer,
    model=ce_model,
    top_k=TOP_K_RERANK,
)
print(f"After cross-encoder: {len(reranked)} candidates")


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 14359.80it/s]


After cross-encoder: 10 candidates


### MMR diversity filter

After reranking, many results can be near-duplicates. **MMR** selects a diverse subset:

\[
\lambda \cdot relevance(d, q) - (1-\lambda) \cdot \max_{s \in selected} sim(d, s)
\]

- Relevance uses `ce_score`.
- Similarity uses cosine similarity between **dense embeddings**.

**Output**: top-`TOP_K_MMR` diverse candidates.


In [13]:
from __future__ import annotations

from typing import Any

import numpy as np


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    denom = (np.linalg.norm(u) * np.linalg.norm(v))
    if denom <= 1e-12:
        return 0.0
    return float(np.dot(u, v) / denom)


def mmr_filter(
    candidates: list[dict[str, Any]],
    dense_encoder: SentenceTransformer,
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
    text_field: str = "parent_text",
) -> list[dict[str, Any]]:
    """Apply MMR to select top_k diverse candidates from reranked list.

    Args:
        candidates: reranked results (should include 'ce_score').
        dense_encoder: used to compute embedding similarity between docs.
        query: user query.
        top_k: number to select.
        lambda_param: relevance/diversity tradeoff.
        text_field: field in metadata to use as text.

    Returns:
        Diverse subset of candidates.
    """
    if not candidates:
        return []

    texts = [_candidate_text(c, text_field) for c in candidates]
    doc_emb = dense_encoder.encode(texts, normalize_embeddings=True)
    doc_emb = np.asarray(doc_emb, dtype=np.float32)

    selected: list[int] = []
    remaining = list(range(len(candidates)))

    # First pick: best ce_score
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)

    while remaining and len(selected) < int(top_k):
        best_idx = None
        best_val = -1e18

        for i in remaining:
            rel = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val = score
                best_idx = i

        assert best_idx is not None
        selected.append(best_idx)
        remaining.remove(best_idx)

    return [candidates[i] for i in selected]


diverse_results = mmr_filter(
    candidates=reranked,
    dense_encoder=dense_encoder,
    query="Is $580k fair for a 4-room in Tampines?",
    top_k=TOP_K_MMR,
    lambda_param=MMR_LAMBDA,
)
print(f"After MMR: {len(diverse_results)} diverse candidates")


After MMR: 5 diverse candidates


### Lost-in-the-middle reordering

LLMs tend to overweight the **start** and **end** of the context window.

We reorder the final set so:
- highest scoring chunk goes first
- second-highest goes last
- the rest fill the middle

**Output**: `final_context` ordered list.


In [14]:
from __future__ import annotations

from typing import Any


def reorder_for_context_window(candidates: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Reorder candidates to mitigate lost-in-the-middle bias.

    Places best first, second-best last.

    Args:
        candidates: list (usually output of MMR), length <= TOP_K_FINAL.

    Returns:
        Reordered list.
    """
    if len(candidates) <= 2:
        return list(candidates)

    # Sort by ce_score if present
    ordered = sorted(candidates, key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)), reverse=True)
    first = ordered[0]
    last = ordered[1]
    middle = ordered[2:]
    return [first, *middle, last]


final_context = reorder_for_context_window(diverse_results)[:TOP_K_FINAL]
for i, c in enumerate(final_context, 1):
    md = c.get("metadata") or {}
    print(f"[slot {i}] {md.get('source')} | {md.get('town')} | {md.get('flat_type')} | {md.get('sale_year')}")


[slot 1] transaction | TAMPINES | 4 ROOM | 2015
[slot 2] transaction | TAMPINES | 4 ROOM | 2015
[slot 3] transaction | TAMPINES | 4 ROOM | 2015
[slot 4] transaction | TAMPINES | 3 ROOM | 2015
[slot 5] transaction | TAMPINES | 4 ROOM | 2015


### Multi-query retrieval strategy

Multi-query retrieval improves recall by generating multiple reformulations of the user query.

We use **Ollama + Gemma 3** to generate `N_SUBQUERIES` alternatives, then:
1. run dense+sparse retrieval per query
2. fuse all ranked lists with RRF
3. rerank + MMR + reorder

**Output**: merged candidate list for reranking.


In [15]:
from __future__ import annotations

from typing import Any, Optional

import re
import ollama


def _set_ollama_host(base_url: str) -> None:
    """Configure Ollama host for the `ollama` python library."""
    # ollama-python reads OLLAMA_HOST
    os.environ["OLLAMA_HOST"] = base_url


def generate_subqueries(query: str, n: int, model: str = OLLAMA_MODEL) -> list[str]:
    """Generate n alternative retrieval queries using Gemma 3.

    Args:
        query: user question.
        n: number of reformulations.
        model: Ollama model name.

    Returns:
        List of reformulated query strings (does not include original).
    """
    _set_ollama_host(OLLAMA_BASE_URL)

    prompt = f"""You are a search query generator for Singapore HDB property data.
Given the user's question, generate {n} alternative search queries that will help retrieve
relevant data from a vector database containing HDB transaction records, amenity proximity,
price trends, and SHAP feature importance data.

Return ONLY the queries as a numbered list. No explanations.

User question: {query}
"""

    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    text = (resp.get("message") or {}).get("content", "")

    # Parse numbered list
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    out: list[str] = []
    for ln in lines:
        m = re.match(r"^\d+[\).\-]\s*(.+)$", ln)
        out.append(m.group(1).strip() if m else ln)
        if len(out) >= int(n):
            break

    # Dedup and remove identical to original
    cleaned = []
    seen = set([query.strip().lower()])
    for q in out:
        k = q.strip().lower()
        if k and k not in seen:
            cleaned.append(q.strip())
            seen.add(k)
    return cleaned[: int(n)]


def multi_query_retrieve(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    n_subqueries: int,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    """Run multi-query retrieval + RRF fusion across all query variants."""
    subqs = generate_subqueries(query, n_subqueries)
    queries = [query, *subqs]

    ranked_lists: list[list[dict[str, Any]]] = []
    for q in queries:
        d = retrieve_dense(index, q, dense_encoder, bm25_encoder, top_k, namespace, metadata_filter)
        s = retrieve_sparse(index, q, dense_encoder, bm25_encoder, top_k, namespace, metadata_filter)
        ranked_lists.append(d)
        ranked_lists.append(s)

    fused = reciprocal_rank_fusion(ranked_lists, k=RRF_K)
    return fused


subqueries = generate_subqueries("Is $580k fair for a 4-room in Tampines?", N_SUBQUERIES)
print("Generated sub-queries:")
for i, q in enumerate(subqueries, 1):
    print(f"  {i}. {q}")


Generated sub-queries:
  1. "HDB 4-room Tampines price trends $580k"
  2. "Tampines HDB 4-room SHAP price $580k"
  3. "Tampines 4-room HDB transaction data amenity proximity $580k"


### Full retrieval pipeline (compose all steps)

This cell composes the full pipeline into `retrieve_and_rerank()`.

This is the function you can lift into the FastAPI backend later.

**Input**: raw user query (+ optional Pinecone metadata filter)

**Output**: top-5 ordered context chunks ready for the LLM.


In [16]:
from __future__ import annotations

from typing import Any, Optional


def retrieve_and_rerank(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    ce_tokenizer: Any,
    ce_model: Any,
    namespace: str,
    metadata_filter: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    """Full retrieval pipeline.

    Steps:
    1. Multi-query fan-out (original + reformulations)
    2. Hybrid retrieval per query (dense + sparse)
    3. RRF fusion across all ranked lists
    4. Cross-encoder reranking (top-50 -> top-10)
    5. MMR diversity (top-10 -> top-5)
    6. Lost-in-middle reordering

    Args:
        query: user question.
        index: Pinecone index.
        dense_encoder: dense bi-encoder.
        bm25_encoder: sparse BM25 encoder.
        ce_tokenizer: cross-encoder tokenizer.
        ce_model: cross-encoder model.
        namespace: Pinecone namespace.
        metadata_filter: optional Pinecone metadata filter.

    Returns:
        Ordered context chunks (list of match dicts).
    """
    # 1-3
    fused = multi_query_retrieve(
        query=query,
        index=index,
        dense_encoder=dense_encoder,
        bm25_encoder=bm25_encoder,
        n_subqueries=N_SUBQUERIES,
        top_k=TOP_K_RETRIEVAL,
        namespace=namespace,
        metadata_filter=metadata_filter,
    )

    # 4
    reranked = rerank_cross_encoder(
        query=query,
        candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer,
        model=ce_model,
        top_k=TOP_K_RERANK,
    )

    # 5
    diverse = mmr_filter(
        candidates=reranked,
        dense_encoder=dense_encoder,
        query=query,
        top_k=TOP_K_MMR,
        lambda_param=MMR_LAMBDA,
    )

    # 6
    final = reorder_for_context_window(diverse)[:TOP_K_FINAL]
    return final


# --- Smoke test ---
query = "Is $580k fair for a 4-room flat in Tampines Street 81?"
context = retrieve_and_rerank(
    query=query,
    index=index,
    dense_encoder=dense_encoder,
    bm25_encoder=bm25_encoder,
    ce_tokenizer=ce_tokenizer,
    ce_model=ce_model,
    namespace=PINECONE_NAMESPACE,
    metadata_filter={"town": "TAMPINES"},
)
print(f"Retrieved {len(context)} context chunks")


Retrieved 5 context chunks


### Prompt builder for Gemma 3

We build:
- a **system prompt** that enforces strict grounding (“answer only from context”), requires citations, and a clear fairness verdict.
- a **user prompt** that injects the retrieved context chunks (with `[Context N]` labels) and the user question.

**Output**: `prompt` string suitable for passing to Ollama.


In [17]:
from __future__ import annotations

from typing import Any


def build_rag_prompt(query: str, context_chunks: list[dict[str, Any]]) -> str:
    """Build the full prompt string for Gemma 3.

    Args:
        query: user question.
        context_chunks: retrieved context chunks.

    Returns:
        Prompt string including labeled context and question.
    """
    parts = []
    parts.append("You are given a set of retrieved context snippets. Use them to answer the question.")
    parts.append("If a claim is not supported by context, do not state it.")
    parts.append("\n## Retrieved context")

    for i, c in enumerate(context_chunks, 1):
        md = c.get("metadata") or {}
        txt = str(md.get("parent_text") or "")
        header = f"[Context {i}] source={md.get('source')} town={md.get('town')} flat_type={md.get('flat_type')} year={md.get('sale_year')}"
        parts.append(header)
        parts.append(txt.strip())
        parts.append("")

    parts.append("## Question")
    parts.append(query)
    return "\n".join(parts).strip()


def build_system_prompt() -> str:
    """Return the system prompt instructing Gemma 3 to stay grounded and cite evidence."""
    system = """You are a Singapore HDB property pricing assistant for the PropertyLens app.
Your job is to help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context. Do not use outside knowledge.
2. Always cite the specific comparable transactions or trend data you used (by [Context N] label).
3. Give a clear price fairness verdict: Fair / Above market / Below market.
4. If the context is insufficient to give a confident answer, say so explicitly.
5. Keep your answer concise: 3–5 sentences max unless the user asks for detail.
"""
    return system


prompt = build_rag_prompt("Is $580k fair for a 4-room in Tampines?", context)
print(prompt[:800], "...")


You are given a set of retrieved context snippets. Use them to answer the question.
If a claim is not supported by context, do not state it.

## Retrieved context
[Context 1] source=transaction town=TAMPINES flat_type=4 ROOM year=2015
This is an HDB resale transaction in TAMPINES, year 2015.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 83.0
Resale price (SGD): 380000
Approx PSF (SGD): 425.3

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA'S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOO ...


### Generate answer with Ollama Gemma 3

We call Ollama locally via the `ollama` Python library.

Prereqs:
- `ollama pull gemma3`
- Ollama running at `localhost:11434`

The call is synchronous. If Ollama is down or the model is missing, the function returns an informative error message.


In [18]:
from __future__ import annotations

from typing import Any


def generate_answer(query: str, context_chunks: list[dict[str, Any]], model: str = OLLAMA_MODEL) -> str:
    """Generate a grounded RAG answer using Ollama + Gemma 3.

    Args:
        query: original user question.
        context_chunks: final reordered context from retrieve_and_rerank().
        model: Ollama model name.

    Returns:
        Answer string.
    """
    _set_ollama_host(OLLAMA_BASE_URL)

    system_prompt = build_system_prompt()
    user_prompt = build_rag_prompt(query, context_chunks)

    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


answer = generate_answer(query, context)
print("=" * 60)
print(f"Query: {query}")
print("=" * 60)
print(answer)


Query: Is $580k fair for a 4-room flat in Tampines Street 81?
I cannot provide a definitive answer with the available context. The data only includes transactions from 2015 for 4-room flats in Tampines, with prices ranging from $369,000 to $380,000, and approximately $408 - $425 PSF [Context 1, 2, 3, 5].  A price of $580k is significantly higher than these recent transactions [Context 1, 2, 3, 5].


### Optional: Generate answer with Google Gemini API

Use this instead of Ollama when you want a cloud model. Set **`GEMINI_API_KEY`** in the repo-root `.env` (same pattern as the backend `chat.py`). Optional override: **`GEMINI_MODEL`** (default `gemini-2.5-flash`, aligned with PropertyLens backend).

**Inputs**: same `query` and `context` from `retrieve_and_rerank()` as the Ollama cell.

**Outputs**: `answer_gemini` string.

Requires: run the pip cell with `google-generativeai` installed.


In [19]:
from __future__ import annotations

from typing import Any

import google.generativeai as genai


def generate_answer_gemini(
    query: str,
    context_chunks: list[dict[str, Any]],
    model: str | None = None,
) -> str:
    """Generate a grounded RAG answer using the Google Gemini API.

    Args:
        query: User question.
        context_chunks: Final context from retrieve_and_rerank().
        model: Gemini model id (defaults to GEMINI_MODEL).

    Returns:
        Answer text, or an error string if the key is missing or the call fails.
    """
    if not GEMINI_API_KEY:
        return "[Gemini error] GEMINI_API_KEY is not set. Add it to repo-root .env"

    name = (model or GEMINI_MODEL).strip()
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        mdl = genai.GenerativeModel(
            model_name=name,
            generation_config=genai.GenerationConfig(
                temperature=0.3,
                max_output_tokens=1024,
            ),
        )
        user_prompt = build_rag_prompt(query, context_chunks)
        full_prompt = f"{build_system_prompt()}\n\n{user_prompt}"
        resp = mdl.generate_content(full_prompt)
        return (getattr(resp, "text", None) or "").strip()
    except Exception as e:
        return f"[Gemini error] {type(e).__name__}: {e}"


# --- Example: reuse query + context from the Ollama smoke-test cell above ---
answer_gemini = generate_answer_gemini(query, context)
print("=" * 60)
print("Gemini answer")
print("=" * 60)
print(answer_gemini)


Gemini answer
Based on the provided context, the resale prices for 4-room flats in Tampines in 2015 ranged from $369,000 to $380,000 [Context 1, 2, 3, 5]. The proposed price of $580,000 is significantly higher than


### End-to-end demo

This runs the full pipeline end-to-end for representative PropertyLens use cases:
- Buyer price fairness
- Seller pricing guidance
- Town trend direction
- Amenities near a specific block (Tampines Street 81, Block 432)

For each query, we print a short context summary plus the generated answer.


In [20]:
from __future__ import annotations

DEMO_QUERIES = [
    {
        "query": "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
        "filter": {"town": "TAMPINES"},
        "persona": "Buyer",
    },
    {
        "query": "What should I list my 5-room Bishan flat for given current market trends?",
        "filter": {"town": "BISHAN"},
        "persona": "Seller",
    },
    {
        "query": "Are HDB prices in Queenstown rising or falling over the last 3 years?",
        "filter": {"town": "QUEENSTOWN"},
        "persona": "Trends",
    },
    {
        "query": "What amenities are near bedok Street 81, Block 432?",
        "filter": {"town": "BEDOK"},
        "persona": "Amenities",
    },
]


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end and print a formatted result."""
    print(f"\n{'='*60}")
    print(f"[{demo['persona']}] {demo['query']}")
    print(f"{'='*60}")

    ctx = retrieve_and_rerank(
        query=demo["query"],
        index=index,
        dense_encoder=dense_encoder,
        bm25_encoder=bm25_encoder,
        ce_tokenizer=ce_tokenizer,
        ce_model=ce_model,
        namespace=PINECONE_NAMESPACE,
        metadata_filter=demo.get("filter"),
    )

    print(f"\nContext chunks retrieved: {len(ctx)}")
    for i, c in enumerate(ctx, 1):
        m = c.get("metadata") or {}
        rp = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"  [{i}] {m.get('source')} | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")

    answer = generate_answer(demo["query"], ctx)
    print(f"\nAnswer:\n{answer}")


for demo in DEMO_QUERIES:
    run_demo(demo)



[Buyer] Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?

Context chunks retrieved: 5
  [1] transaction | TAMPINES | 4 ROOM | $380,000 | 2015
  [2] transaction | TAMPINES | 4 ROOM | $369,000 | 2015
  [3] transaction | TAMPINES | 4 ROOM | $370,000 | 2015
  [4] transaction | TAMPINES | 3 ROOM | $410,000 | 2015
  [5] transaction | TAMPINES | 4 ROOM | $378,000 | 2015

Answer:
I cannot provide a definitive answer with the available context. The context includes several 4-room HDB transactions in Tampines from 2015, ranging in price from $369,000 to $378,000 with PSF values between $351.2 and $425.3. [Context 1], [Context 2], [Context 3], [Context 5]  Given this limited data, I cannot assess the fairness of $580k.

[Seller] What should I list my 5-room Bishan flat for given current market trends?

Context chunks retrieved: 5
  [1] transaction | BISHAN | 5 ROOM | $820,000 | 2015
  [2] transaction | BISHAN | 5 ROOM | $590,000 | 2015
  [3] transaction | BISHAN | 5 ROOM | $851,0

### Known limitations and next steps

#### Known limitations
- School quality not included — amenities contain proximity, not rankings/popularity.
- Trends chunks are derived from the same transaction data — no external market signals.
- Gemma 3 (local) is smaller than GPT-4 class models; complex multi-hop reasoning may be weaker.
- Pinecone free tier has limits on index size — may need paid tier for full transaction history.

#### Suggested next steps
- Add MOE school popularity data (Phase 2A oversubscription rates) to amenities CSVs.
- Fine-tune BGE-M3 embeddings on PropertyLens domain queries to improve retrieval precision.
- Expose `retrieve_and_rerank()` as a new FastAPI endpoint `/api/rag/query` in `backend/main.py`.
- Evaluate retrieval quality with a labelled test set using NDCG@5.
